# HARs Burden Test - SAS (South Asian) ancestry only - Novalis

Corre burden tests con RVTests sobre **ancestría SAS** (South Asian), partiendo de los VCFs por-HAR ya generados por `regionExtractor.py` (carpeta `Indiv_HARS/`).

**Cohorte SAS** (GP2 R11):
- NBA: 1,290 muestras (412 PD / 405 Control / 473 Other)
- WGS: 986 muestras (240 PD / 344 Control / 402 Other)

Cohorte chica → sin riesgo de OOM, corre rápido, balance PD/Control razonable.

**Decisiones de diseño**:
- **No se corre ANNOVAR**. Las HARs son no-codificantes; las clases dependientes de anotación funcional (`coding_variants`, `lof`, `coding_nonsynonymous`, `potentially_functional`) y `potencially_pathogenic` (CLNSIG/REVEL/CADD) no aplican o están sesgadas hacia variantes codificantes.
- **MAF filtering nativo en RVTests** vía `--freqUpper`, usando MAF interna de la cohorte. Evita ANNOVAR.
- **Tres clases de burden**: `all_variants`, `maf01` (MAF<0.01), `maf03` (MAF<0.03).
- **Custom HAR-refFlat**: cada HAR figura como un "gen" con sus coordenadas para `--gene HAR --geneFile`.
- **Solo HARs con VCF disponible** (filtrado vía `present_hars`).
- **Resume + retry**: si el `.assoc` ya existe lo saltea (`[CACHED]`); si rvtest muere por SIGKILL (OOM), reintenta una vez después de 5 seg.

**Outputs**:
- Por-HAR: `/home/jupyter/workspace/ws_files/Novalis/Burden/{set}/SAS/{HAR}_{class}.burden.{Skat,SkatO}.assoc`
- Compilados: `/home/jupyter/workspace/ws_files/Novalis/Results/{set}_SAS_BURDEN.{SKAT,SKAT-O}.tsv`

## 0. Instalar RVTests si no está

Si el binario ya existe, no hace nada. Si no, lo descarga e instala (~1-3 min).

In [ ]:
%%bash
if test -e /home/jupyter/tools/rvtests/executable/rvtest; then
    echo "RVtests ya está instalado:"
    ls -la /home/jupyter/tools/rvtests/executable/rvtest
else
    echo "Instalando RVtests..."
    mkdir -p /home/jupyter/tools/rvtests
    cd /home/jupyter/tools/rvtests
    wget -q https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz
    tar -zxvf rvtests_linux64.tar.gz
    chmod 777 /home/jupyter/tools/rvtests/executable/rvtest
    echo ""
    echo "Listo:"
    ls -la /home/jupyter/tools/rvtests/executable/rvtest
fi

## 1. Setup, imports y configuración

In [ ]:
import os
import csv
import time
import pathlib
import subprocess
from datetime import datetime
from multiprocessing import Manager
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
from pathlib import Path

In [ ]:
# ---------- Configuración ----------
release = 11

# >>> SAS ONLY <<<
ANCESTRY = "SAS"

# Datasets a correr (SAS tiene ambos disponibles)
datasets = ["NBA", "WGS"]

# Clases de burden (sin potencially_pathogenic)
BURDEN_CLASSES = {
    "all_variants": None,    # sin filtro de MAF
    "maf01":        0.01,    # --freqUpper 0.01
    "maf03":        0.03,    # --freqUpper 0.03
}

# Paralelismo: cohort chica, sin riesgo OOM. Igual al N° de CPUs de la VM.
MAX_WORKERS = 16

# ---------- Paths ----------
DIR_TOOL  = "/home/jupyter/tools"
DIR_WSPS  = "/home/jupyter/workspace/ws_files"
DIR_ORIG  = f"{DIR_WSPS}/Original_Files"
DIR_NOVA  = f"{DIR_WSPS}/Novalis"
DIR_BURDEN = f"{DIR_NOVA}/Burden"
DIR_RESU   = f"{DIR_NOVA}/Results"

RVTEST  = f"{DIR_TOOL}/rvtests/executable/rvtest"
PLINK2  = f"{DIR_TOOL}/plink2"

# refFlat custom donde cada HAR es un "gen"
HAR_REFFLAT = f"{DIR_NOVA}/HAR_refFlat_hg38.txt"

# Crear directorios base
for d in [DIR_NOVA, DIR_BURDEN, DIR_RESU]:
    os.makedirs(d, exist_ok=True)
for ds in datasets:
    os.makedirs(f"{DIR_BURDEN}/{ds}/{ANCESTRY}", exist_ok=True)

print(f"Ancestry:    {ANCESTRY}")
print(f"Datasets:    {datasets}")
print(f"Burden classes: {list(BURDEN_CLASSES.keys())}")
print(f"MAX_WORKERS: {MAX_WORKERS}")
print(f"Output root: {DIR_NOVA}")
print(f"RVTest:      {RVTEST}")

## 2. Cargar lista de HARs

In [ ]:
with open(f'{DIR_ORIG}/HAR_list_phase_1.tsv', 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    HARS_DICT = {
        row[3]: {
            'name':  row[3],
            'chrom': row[0].replace('chr', ''),
            'start': int(row[1]),
            'end':   int(row[2]),
        }
        for row in reader
    }
print(f"HARs cargados: {len(HARS_DICT)}")
print("Ejemplo:", next(iter(HARS_DICT.items())))

## 3. Construir refFlat custom para HARs

RVTests usa `--gene NAME --geneFile refFlat` para agrupar variantes. Como las HARs no aparecen en el refFlat estándar, generamos uno propio donde cada HAR es un "gen" cuyas coordenadas cubren toda la región.

**Formato refFlat** (11 columnas, tab-separated):
`geneName  name  chrom  strand  txStart  txEnd  cdsStart  cdsEnd  exonCount  exonStarts  exonEnds`

In [ ]:
with open(HAR_REFFLAT, 'w') as f:
    for HAR, info in HARS_DICT.items():
        chrom = info['chrom'] if info['chrom'].startswith('chr') else f"chr{info['chrom']}"
        start = info['start']
        end   = info['end']
        f.write('\t'.join([
            HAR, HAR, chrom, '+',
            str(start), str(end),
            str(start), str(end),
            '1',
            f'{start},',
            f'{end},'
        ]) + '\n')

print(f"HAR refFlat escrito en: {HAR_REFFLAT}")
!head -3 {HAR_REFFLAT}
!wc -l {HAR_REFFLAT}

## 4. Verificar inputs y construir lista `present_hars`

Verifica que existan los `.vcf.gz`/`.tbi` por HAR y el covariate file para SAS. Lista el directorio una sola vez por dataset (rápido) y guarda `present_hars[ds]` con los HARs analizables, que se usa en la celda 6 para evitar tareas inútiles.

In [ ]:
def vcf_path(set_name, HAR):
    return f"{DIR_WSPS}/Working_{set_name}/{ANCESTRY}/InputFiles/Indiv_HARS/{HAR}.vcf.gz"

def covar_path(set_name):
    return f"{DIR_WSPS}/Working_{set_name}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"

summary = {}
present_hars = {}    # HARs analizables por dataset (con VCF + tbi)

for ds in datasets:
    cov = covar_path(ds)
    cov_ok = os.path.isfile(cov)

    indiv_dir = f"{DIR_WSPS}/Working_{ds}/{ANCESTRY}/InputFiles/Indiv_HARS"
    try:
        files = set(os.listdir(indiv_dir))
    except FileNotFoundError:
        files = set()

    present = [HAR for HAR in HARS_DICT
               if f"{HAR}.vcf.gz" in files and f"{HAR}.vcf.gz.tbi" in files]
    present_set = set(present)
    missing = [HAR for HAR in HARS_DICT if HAR not in present_set]

    present_hars[ds] = present

    summary[ds] = {
        'covariate_ok':    cov_ok,
        'covariate_path':  cov,
        'indiv_dir':       indiv_dir,
        'n_files_in_dir':  len(files),
        'n_vcfs_present':  len(present),
        'n_vcfs_missing':  len(missing),
        'missing_examples': missing[:5],
    }

for ds, s in summary.items():
    print(f"\n=== {ds} ===")
    for k, v in s.items():
        print(f"  {k}: {v}")

**Nota**: los HARs en `n_vcfs_missing` son los que `regionExtractor.py` marcó como SKIP (sin variantes que pasen filtros). Están en `Working_{set}/{ANCESTRY}/failed_HARs_{set}_{ANCESTRY}.tsv`. Esos HARs no se van a testear, es esperado.

## 5. Función de burden test (con resume + retry para OOM)

- **Resume**: si los dos `.assoc` ya existen y no están vacíos, marca `[CACHED]` sin re-correr (útil si reanudás).
- **Retry SIGKILL**: si rvtest muere con `rc=-9` (OOM killer), espera 5 seg y reintenta una vez.
- Una sola función; clase MAF determinada por `freq_upper` (`None` = sin filtro).

In [ ]:
def burden_rvtest(set_name, HAR, burden_class, freq_upper):
    """Corre RVTests SKAT/SKAT-O con resume y retry para SIGKILL."""
    in_vcf  = vcf_path(set_name, HAR)
    covar   = covar_path(set_name)

    if not os.path.isfile(in_vcf):
        return f"[SKIP] {set_name} {HAR} {burden_class} - no input VCF"

    out_dir = f"{DIR_BURDEN}/{set_name}/{ANCESTRY}"
    os.makedirs(out_dir, exist_ok=True)
    out_prefix = f"{out_dir}/{HAR}_{burden_class}.burden"

    # RESUME: si los dos .assoc ya existen y no están vacíos, saltar
    skat_done  = os.path.isfile(f"{out_prefix}.Skat.assoc")  and os.path.getsize(f"{out_prefix}.Skat.assoc")  > 0
    skato_done = os.path.isfile(f"{out_prefix}.SkatO.assoc") and os.path.getsize(f"{out_prefix}.SkatO.assoc") > 0
    if skat_done and skato_done:
        return f"[CACHED] {set_name} {HAR} {burden_class}"

    cmd = [
        RVTEST,
        "--noweb", "--hide-covar",
        "--inVcf",     in_vcf,
        "--out",       out_prefix,
        "--kernel",    "skat,skato",
        "--pheno",     covar,
        "--pheno-name","PHENO",
        "--covar",     covar,
        "--covar-name","SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10",
        "--gene",      HAR,
        "--geneFile",  HAR_REFFLAT,
    ]
    if freq_upper is not None:
        cmd += ["--freqUpper", str(freq_upper)]

    # RETRY: hasta 2 intentos si rc=-9 (SIGKILL/OOM)
    for attempt in (1, 2):
        ts = datetime.now().strftime('%H:%M:%S')
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=False)
            if result.returncode == 0:
                tag = " (retry)" if attempt == 2 else ""
                return f"[DONE] {ts} {set_name} {HAR} {burden_class}{tag}"
            if result.returncode == -9 and attempt == 1:
                # OOM kill: esperar 5 seg y reintentar
                time.sleep(5)
                continue
            return f"[FAIL] {ts} {set_name} {HAR} {burden_class} | rc={result.returncode} | {result.stderr[-150:]}"
        except Exception as e:
            return f"[ERR ] {ts} {set_name} {HAR} {burden_class} | {e}"
    return f"[FAIL2] {ts} {set_name} {HAR} {burden_class} | OOM x2"


def _run_task(args):
    return burden_rvtest(*args)

## 6. Correr burden tests para SAS (paralelo, solo HARs presentes)

In [ ]:
# Construir lista de tareas: (set, HAR, class, freqUpper)
tasks = [
    (ds, HAR, cls_name, cls_freq)
    for ds in datasets
    for HAR in present_hars[ds]
    for cls_name, cls_freq in BURDEN_CLASSES.items()
]

print(f"Total tareas: {len(tasks)}")
for ds in datasets:
    n = len(present_hars[ds]) * len(BURDEN_CLASSES)
    print(f"  {ds}: {len(present_hars[ds])} HARs x {len(BURDEN_CLASSES)} clases = {n} tareas")
print(f"Workers: {MAX_WORKERS}\n")

ts_start = datetime.now()
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(_run_task, t): t for t in tasks}
    completed = 0
    n_done = n_cached = n_fail = n_skip = 0
    for fut in as_completed(futures):
        completed += 1
        try:
            msg = fut.result()
        except Exception as e:
            msg = f"[EXC ] {futures[fut]} | {e}"

        # Contadores
        if   msg.startswith('[DONE'):   n_done   += 1
        elif msg.startswith('[CACHED'): n_cached += 1
        elif msg.startswith('[SKIP'):   n_skip   += 1
        else:                            n_fail   += 1

        # Imprimir cada 100 + siempre los errores
        if completed % 100 == 0 or msg.startswith(('[FAIL', '[ERR ', '[EXC ')):
            elapsed = datetime.now() - ts_start
            rate = completed / max(elapsed.total_seconds() / 60, 0.01)
            eta_min = (len(tasks) - completed) / max(rate, 0.01)
            print(f"[{completed}/{len(tasks)}] D={n_done} C={n_cached} F={n_fail} S={n_skip} | {rate:.1f}/min | ETA {eta_min/60:.1f}h | {msg}")

print(f"\nTerminado en {datetime.now() - ts_start}")
print(f"Resumen: DONE={n_done}  CACHED={n_cached}  FAIL={n_fail}  SKIP={n_skip}")

## 7. Compilar resultados (SKAT y SKAT-O)

Junta todos los `.assoc` por dataset/clase en dos TSVs, agregando columnas ANCESTRY, GENE (HAR), CLASS, DATASET.

In [ ]:
def collect_assoc(set_name):
    skat_rows  = []
    skato_rows = []
    burden_dir = Path(f"{DIR_BURDEN}/{set_name}/{ANCESTRY}")

    for HAR in present_hars[set_name]:
        for cls in BURDEN_CLASSES:
            p_skat  = burden_dir / f"{HAR}_{cls}.burden.Skat.assoc"
            p_skato = burden_dir / f"{HAR}_{cls}.burden.SkatO.assoc"

            if p_skat.is_file():
                try:
                    df = pd.read_csv(p_skat, sep='\t')
                    df.insert(0, 'ANCESTRY', ANCESTRY)
                    df.insert(1, 'GENE', HAR)
                    df.insert(2, 'CLASS', cls)
                    df.insert(3, 'DATASET', set_name)
                    skat_rows.append(df)
                except Exception as e:
                    print(f"  warn read SKAT {HAR} {cls}: {e}")

            if p_skato.is_file():
                try:
                    df = pd.read_csv(p_skato, sep='\t')
                    df.insert(0, 'ANCESTRY', ANCESTRY)
                    df.insert(1, 'GENE', HAR)
                    df.insert(2, 'CLASS', cls)
                    df.insert(3, 'DATASET', set_name)
                    skato_rows.append(df)
                except Exception as e:
                    print(f"  warn read SKAT-O {HAR} {cls}: {e}")

    skat_df  = pd.concat(skat_rows,  ignore_index=True) if skat_rows  else pd.DataFrame()
    skato_df = pd.concat(skato_rows, ignore_index=True) if skato_rows else pd.DataFrame()
    return skat_df, skato_df


for ds in datasets:
    print(f"\n=== Compilando {ds} ===")
    skat_df, skato_df = collect_assoc(ds)
    out_skat  = f"{DIR_RESU}/{ds}_{ANCESTRY}_BURDEN.SKAT.tsv"
    out_skato = f"{DIR_RESU}/{ds}_{ANCESTRY}_BURDEN.SKAT-O.tsv"
    skat_df.to_csv(out_skat,   sep='\t', index=False)
    skato_df.to_csv(out_skato, sep='\t', index=False)
    print(f"  SKAT   : {len(skat_df):>5} filas → {out_skat}")
    print(f"  SKAT-O : {len(skato_df):>5} filas → {out_skato}")

## 8. Quick look: top hits

In [ ]:
for ds in datasets:
    for kernel in ['SKAT', 'SKAT-O']:
        path = f"{DIR_RESU}/{ds}_{ANCESTRY}_BURDEN.{kernel}.tsv"
        if not os.path.isfile(path):
            continue
        df = pd.read_csv(path, sep='\t')
        if df.empty or 'Pvalue' not in df.columns:
            continue
        print(f"\n--- Top 10 {ds} {kernel} ---")
        print(df.sort_values('Pvalue').head(10).to_string(index=False))